# Hyperparameter Tuning

We train the same CNN with a few different settings and keep the one with the best **validation** accuracy.
The best settings are saved to `models/best_hparams.json`, and `model.ipynb` uses them automatically.

The test set is never used here.

## 1. Imports

In [ ]:
import sys, json
from pathlib import Path
root = Path.cwd().parent
sys.path.append(str(root))

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

import src.training.pretraining as Pretrain
import src.training.tuning as Tuner


from src.models.cnn import build_model

I0000 00:00:1792062976.979122  190220 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792062976.985161  190220 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792062977.631115  190220 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792062981.160530  190220 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

TensorFlow: 2.21.0
GPUs: []


E0000 00:00:1792062983.410833  190220 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 2. Load the training data and create a validation set

Use the same split as in `model.ipynb` (same `random_state`), so results are comparable.

In [2]:
X_train, X_test, y_train, y_test, y_train_cat, y_test_cat, encoder, class_names = Pretrain.load()

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_cat, test_size=0.15, stratify=y_train, random_state=42
)

print("Train:", X_tr.shape, "| Validation:", X_val.shape)

X_train: (4916, 224, 224, 3) | y_train_cat: (4916, 4)
X_test : (2107, 224, 224, 3) | y_test_cat : (2107, 4)
Classes: [np.str_('glioma'), np.str_('meningioma'), np.str_('notumor'), np.str_('pituitary')]
Pixel range: 0.0 1.0
Train: (4178, 224, 224, 3) | Validation: (738, 224, 224, 3)


## 3. Choose the settings to try

The first one is the baseline. Each of the others changes one thing.


In [ ]:
configs = [
    {"name": "baseline",    "learning_rate": 1e-3, "dropout": 0.5, "dense_units": 256, "batch_size": 32},
    {"name": "lr 3e-4",     "learning_rate": 3e-4, "dropout": 0.5, "dense_units": 256, "batch_size": 32},
    {"name": "lr 3e-3",     "learning_rate": 3e-3, "dropout": 0.5, "dense_units": 256, "batch_size": 32},
    {"name": "dropout 0.3", "learning_rate": 1e-3, "dropout": 0.3, "dense_units": 256, "batch_size": 32},
    {"name": "units 128",   "learning_rate": 1e-3, "dropout": 0.5, "dense_units": 128, "batch_size": 32},
    {"name": "batch 16",    "learning_rate": 1e-3, "dropout": 0.5, "dense_units": 256, "batch_size": 16},
]

EPOCHS = 10

## 4. Train each configuration

Each run starts from the same seed. Early stopping ends a run when the validation loss stops improving.
This takes a while: use a GPU if you can.

In [ ]:
results = Tuner.train(configs, X_tr, y_tr, X_val, y_val, EPOCHS)

## 5. Compare the results

In [ ]:
results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
results_df

## 6. Save the best settings

In [ ]:
best = results_df.iloc[0]

best_params = {
    "learning_rate": float(best["learning_rate"]),
    "dropout": float(best["dropout"]),
    "dense_units": int(best["dense_units"]),
    "batch_size": int(best["batch_size"]),
}

(root / "models").mkdir(exist_ok=True)
(root / "models" / "best_hparams.json").write_text(json.dumps(best_params, indent=2))

print("Best configuration:", best["name"])
print(best_params)